In [0]:
dbutils.widgets.text("target_catalog", "tpch_dev", "Target Catalog")
target_catalog = dbutils.widgets.get("target_catalog")
print(f"Building gold layer for catalog: {target_catalog}")

In [0]:
from pyspark.sql import functions as F

orders = spark.table(f"{target_catalog}.silver.orders")
lineitem = spark.table(f"{target_catalog}.silver.lineitem")
customer = spark.table(f"{target_catalog}.silver.customer")
nation = spark.table(f"{target_catalog}.silver.nation")
region = spark.table(f"{target_catalog}.silver.region")

revenue_by_region_quarter = (
    lineitem
    .join(orders, lineitem.l_orderkey == orders.o_orderkey)
    .join(customer, orders.o_custkey == customer.c_custkey)
    .join(nation, customer.c_nationkey == nation.n_nationkey)
    .join(region, nation.n_regionkey == region.r_regionkey)
    .withColumn("quarter", F.date_trunc("quarter", "o_orderdate"))
    .withColumn("net_revenue", F.col("l_extendedprice") * (1 - F.col("l_discount")))
    .groupBy("r_name", "quarter")
    .agg(
        F.sum("net_revenue").alias("total_revenue"),
        F.count("*").alias("line_item_count")
    )
    .orderBy("r_name", "quarter")
)

revenue_by_region_quarter.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{target_catalog}.gold.revenue_by_region_quarter")

print(f"Gold table created: {target_catalog}.gold.revenue_by_region_quarter")
revenue_by_region_quarter.display()

In [0]:
revenue_by_region_quarter.display()

In [0]:
supplier = spark.table(f"{target_catalog}.silver.supplier")
part = spark.table(f"{target_catalog}.silver.part")
nation_s = spark.table(f"{target_catalog}.silver.nation")

supplier_performance = (
    lineitem
    .join(F.broadcast(supplier), lineitem.l_suppkey == supplier.s_suppkey)
    .join(F.broadcast(nation_s), supplier.s_nationkey == nation_s.n_nationkey)
    .withColumn("net_revenue", F.col("l_extendedprice") * (1 - F.col("l_discount")))
    .groupBy("s_suppkey", "s_name", "n_name")
    .agg(
        F.sum("net_revenue").alias("total_revenue"),
        F.count("*").alias("order_line_count")
    )
    .orderBy(F.col("total_revenue").desc())
)

supplier_performance.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{target_catalog}.gold.supplier_performance")

print(f"Gold table created: {target_catalog}.gold.supplier_performance")
supplier_performance.limit(10).display()

In [0]:
supplier_performance.limit(10).display()

In [0]:
spark.sql(f"OPTIMIZE {target_catalog}.gold.revenue_by_region_quarter ZORDER BY (r_name)")
spark.sql(f"OPTIMIZE {target_catalog}.gold.supplier_performance ZORDER BY (s_suppkey)")

In [0]:
spark.sql("SHOW TABLES IN tpch_dev.gold").display()